## Embeddings using torch

In [1]:
import numpy as np
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

In [7]:
with open("data.txt", "r") as file:
    raw_text = file.read()

In [8]:
class DreamLLMDataset(Dataset):
    def __init__(self, raw_text, tokenizer, context_length=10, stride=1):

        # create the tokens
        tokens =  tokenizer.encode(raw_text)

        # store the input and target tensors
        self.input_ids = []
        self.target_ids = []

        # create the input and target tensors
        for i in range(0, len(tokens)-context_length,stride):
            input_chunk = tokens[i : i + context_length]
            target_chunk = tokens[i + 1 : i + context_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]        


In [9]:
# create the tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

In [10]:
# create the dataset 
dataset  = DreamLLMDataset(raw_text=raw_text, tokenizer=tokenizer)

# create a dataloader
data_loader = DataLoader(dataset=dataset, batch_size=5)

In [12]:
# get the iterator
data_loader_iterator = iter(data_loader)

# get the first batch
inputs, targets = next(data_loader_iterator)

In [13]:
inputs, targets

(tensor([[44484,   373,  3726,   284,   651,   845, 10032,   286,  5586,   416],
         [  373,  3726,   284,   651,   845, 10032,   286,  5586,   416,   607],
         [ 3726,   284,   651,   845, 10032,   286,  5586,   416,   607,  6621],
         [  284,   651,   845, 10032,   286,  5586,   416,   607,  6621,   198],
         [  651,   845, 10032,   286,  5586,   416,   607,  6621,   198,   261]]),
 tensor([[  373,  3726,   284,   651,   845, 10032,   286,  5586,   416,   607],
         [ 3726,   284,   651,   845, 10032,   286,  5586,   416,   607,  6621],
         [  284,   651,   845, 10032,   286,  5586,   416,   607,  6621,   198],
         [  651,   845, 10032,   286,  5586,   416,   607,  6621,   198,   261],
         [  845, 10032,   286,  5586,   416,   607,  6621,   198,   261,   262]]))

## Create an embedding layer using pytorch

In [15]:
# get the vocab size
vocab_size = tokenizer.n_vocab

# declare the embedding dimensions
dimensions = 10

In [16]:
# create the embedding layer
embedding_layer = torch.nn.Embedding(vocab_size, dimensions)
embedding_layer.weight

Parameter containing:
tensor([[-0.7493, -0.2282, -1.8372,  ..., -0.2821,  0.1916, -0.1402],
        [-0.4264, -0.1688,  0.0397,  ..., -0.0649, -0.7155, -0.4613],
        [ 0.3294, -0.5344, -1.4954,  ..., -0.7932,  0.1411,  1.0819],
        ...,
        [-1.0804,  0.2662, -1.9889,  ...,  0.1336,  0.5235,  1.4393],
        [-0.2173,  1.2482, -0.2589,  ...,  1.8251,  0.2239,  0.0786],
        [-2.0066, -1.8521, -0.5379,  ..., -1.0494, -0.1036,  2.1673]],
       requires_grad=True)

In [17]:
input_embeddings = embedding_layer(inputs)
input_embeddings

tensor([[[-3.7418e-01, -3.0413e-01,  1.2028e+00, -1.6093e+00, -2.9968e-01,
          -5.1708e-02, -5.5563e-01,  2.3572e-01, -1.4325e+00,  6.2589e-01],
         [-1.4187e+00, -9.4041e-01, -1.1984e+00, -9.3035e-01,  2.4285e-01,
           9.5785e-02,  1.5108e+00,  6.9632e-01, -1.6384e+00, -7.1147e-01],
         [ 2.3361e+00,  6.2560e-01,  6.1820e-01,  1.3514e+00, -6.0998e-01,
          -2.0127e-01, -9.6672e-01, -7.4422e-01, -1.8629e+00, -1.4495e+00],
         [ 2.1514e-01, -4.8624e-01, -1.4475e+00,  5.6479e-01, -6.0423e-01,
          -2.3949e+00,  8.2888e-01, -6.3583e-01, -1.4870e+00, -6.0217e-01],
         [-3.1761e-01,  1.2477e+00,  1.3378e-01, -8.5400e-02,  3.7574e-01,
          -3.7160e-01, -1.3112e+00,  3.1566e-01,  1.1170e+00,  1.5083e+00],
         [ 1.0140e+00,  2.3260e-01,  2.3485e+00,  1.3605e+00,  2.1906e-01,
           8.0213e-01, -3.8456e-01, -1.1824e-01,  9.5698e-01,  6.7769e-01],
         [ 1.0575e+00, -2.3012e-03,  2.8946e-01, -1.8292e+00,  3.6577e-01,
          -7.4569e-

In [18]:
input_embeddings.shape

torch.Size([5, 10, 10])